In [1]:
!pip install -q kagglehub tensorflow tqdm

In [2]:
import kagglehub

path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-dataset")
print("Path to dataset files:", path)

100%|██████████| 23.1G/23.1G [15:50<00:00, 26.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/paramaggarwal/fashion-product-images-dataset/versions/1


In [3]:
import os

# Recursively find every jpg under the downloaded dataset — structure varies by version
all_images = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            all_images.append(os.path.join(root, f))

print(f"Found {len(all_images)} images")
print("Example path:", all_images[0])

Found 88882 images
Example path: /root/.cache/kagglehub/datasets/paramaggarwal/fashion-product-images-dataset/versions/1/fashion-dataset/images/50667.jpg


In [4]:
import random

random.seed(42)
SAMPLE_SIZE = 3000  # tune this — bigger = more variety, bigger repo
sampled_images = random.sample(all_images, min(SAMPLE_SIZE, len(all_images)))
print(f"Sampled {len(sampled_images)} images")

Sampled 3000 images


In [5]:
from PIL import Image
import shutil

os.makedirs("images", exist_ok=True)

MAX_SIDE = 300  # resize longest side to this, keeps files small but still viewable

final_filenames = []
for src_path in sampled_images:
    fname = os.path.basename(src_path)
    dst_path = os.path.join("images", fname)
    try:
        img = Image.open(src_path).convert("RGB")
        img.thumbnail((MAX_SIDE, MAX_SIDE))
        img.save(dst_path, "JPEG", quality=85)
        final_filenames.append(dst_path)
    except Exception as e:
        print(f"Skipped {src_path}: {e}")

print(f"Saved {len(final_filenames)} resized images to ./images")

Saved 3000 resized images to ./images


In [6]:
import tensorflow
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import GlobalMaxPooling2D
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
import numpy as np
from numpy.linalg import norm
from tqdm import tqdm
import pickle

model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
model.trainable = False
model = tensorflow.keras.Sequential([model, GlobalMaxPooling2D()])

def extract_features(img_path, model):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    expanded_img_array = np.expand_dims(img_array, axis=0)
    preprocessed_img = preprocess_input(expanded_img_array)
    result = model.predict(preprocessed_img, verbose=0).flatten()
    return result / norm(result)

feature_list = []
valid_filenames = []

for fpath in tqdm(final_filenames):
    try:
        feature_list.append(extract_features(fpath, model))
        valid_filenames.append(fpath)  # e.g. 'images/12345.jpg' — matches repo layout
    except Exception as e:
        print(f"Skipped {fpath}: {e}")

pickle.dump(feature_list, open('embeddings.pkl', 'wb'))
pickle.dump(valid_filenames, open('filenames.pkl', 'wb'))
print("Saved embeddings.pkl and filenames.pkl")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


100%|██████████| 3000/3000 [17:54<00:00,  2.79it/s]


Saved embeddings.pkl and filenames.pkl


In [7]:
import shutil

shutil.make_archive("fashion_recommender_data", "zip", ".", "images")  # zips the images/ folder

# Move pkl files alongside for easy download too
from google.colab import files
files.download("embeddings.pkl")
files.download("filenames.pkl")
files.download("fashion_recommender_data.zip")  # extract this into your repo root as `images/`

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>